# Agentic RAG Challenge -- Student Academic Intelligence System

---

## Objective

Build an **Agentic RAG pipeline** that answers 50 test queries about student performance by reasoning over structured data (CSVs) and unstructured policy documents (Markdown).

## Submission Format

| Column | Type | Description |
|--------|------|-------------|
| `answerable` | 0 or 1 | 1 = answerable, 0 = out-of-scope |
| `lookup_value` | float | Raw data value |
| `rule_value` | float | Policy rule applied (0 if no rule needed) |
| `final_answer` | float | Computed result |

**Out-of-scope:** `answerable=0, lookup_value=-1, rule_value=-1, final_answer=-1`

**Evaluation:** RMSE across all 4 columns. Lower is better.

---

> Follow the cells sequentially. Complete sections marked TODO.
>
> **IMPORTANT:** The submission generation cell (Cell 11) MUST be executed for your notebook to produce a valid submission on Kaggle.
>
> **Rate limits:** The notebook ships with a `KeyPool` that spreads calls across your Groq keys and cools down any key that hits a 429. Add **at least 3-4 Groq keys** as Kaggle Secrets -- 50 queries then finish in roughly 10 minutes. Progress is saved to `submission_partial.csv` every 10 queries.

## Cell 1 -- Install Dependencies

> Run this cell first. After installation, restart the kernel and run all cells from the top.

In [ ]:
print("Installing required packages...\n")

!pip install -q \
    openai \
    google-genai \
    faiss-cpu \
    pandas \
    numpy

print("\nInstallation complete.")
print("Please restart the kernel/runtime before running the next cell.")

## Cell 2 -- Configuration

Set your data path and model preferences here. All remaining cells use this configuration.

In [ ]:
# -- Configuration --
import os

# Auto-detect data path (supports both local VS Code workspace and Kaggle runtime)
if os.path.exists("Dataset"):
    DATA_PATH = "."
elif os.path.exists("/kaggle/input/student-performance-parent-call-engine"):
    DATA_PATH = "/kaggle/input/student-performance-parent-call-engine"
else:
    DATA_PATH = "."

# LLM & Embedding Model configurations
# llama-3.3-70b-versatile is the current premier Groq model (high throughput, excellent tool-use)
LLM_MODEL = "llama-3.3-70b-versatile"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
REF_DATE = "2025-01-15"

print(f"Data path: {DATA_PATH}")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Reference Date: {REF_DATE}")


## Cell 3 -- Imports

> Run this cell without modification.

In [ ]:
import os, json, re, time, csv, math
from datetime import datetime, date
from collections import defaultdict
import numpy as np
import pandas as pd
from openai import OpenAI

# Google GenAI (optional)
try:
    from google import genai
except Exception:
    genai = None

# Vector indexing & similarity modules
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Imports loaded successfully.")


## Cell 4 -- API Keys & Rate Limiting

Add your API keys as **Kaggle Secrets** before running this cell (Notebook menu > **Add-ons > Secrets**):
- `GROQ_API_KEY_1`, `GROQ_API_KEY_2`, ... up to `_8` (**use at least 3-4 keys**)
- `GOOGLE_API_KEY`

**Important:** after adding each secret, make sure its toggle is **ON (Attached)** for this notebook, and that **Internet** is enabled in the notebook settings.

`KeyPool` picks a key that is under its per-minute limit and puts any key that returns a 429 on cooldown.


In [ ]:
# ---- Load API keys from Kaggle Secrets (or local environment / interactive input) ----
import getpass, sys

def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        v = secrets.get_secret(name)
        if v:
            return v.strip()
    except Exception:
        pass
    return os.environ.get(name)

# Groq keys: GROQ_API_KEY_1 ... GROQ_API_KEY_8 (gaps skipped, duplicates removed)
GROQ_KEYS = []
for i in range(1, 9):
    k = _secret(f"GROQ_API_KEY_{i}")
    if k and k not in GROQ_KEYS:
        GROQ_KEYS.append(k)

# Also check standard GROQ_API_KEY
single_key = _secret("GROQ_API_KEY")
if single_key and single_key not in GROQ_KEYS:
    GROQ_KEYS.append(single_key)

GOOGLE_API_KEY = _secret("GOOGLE_API_KEY")

# Interactive prompt fallback only if running in an interactive terminal
if not GROQ_KEYS:
    print("Notice: No Groq API keys found in Kaggle Secrets or environment.")
    print("In Kaggle: Add-ons > Secrets, add GROQ_API_KEY_1 and attach to notebook.")
    if sys.stdin and hasattr(sys.stdin, "isatty") and sys.stdin.isatty():
        try:
            entered = getpass.getpass("Enter your Groq API key (or press Enter to run offline): ").strip()
            if entered:
                GROQ_KEYS.append(entered)
        except Exception:
            pass

print(f"API configuration loaded -- {len(GROQ_KEYS)} Groq key(s) active.")
if 0 < len(GROQ_KEYS) < 3:
    print("TIP: Having 3-4 Groq keys attached helps avoid 429 rate-limit waits.")

# Gemini client (optional for embeddings)
gemini_client = None
if GOOGLE_API_KEY:
    try:
        from google import genai
        gemini_client = genai.Client(api_key=GOOGLE_API_KEY)
        print("Gemini client initialized for embeddings.")
    except Exception as e:
        print(f"Gemini notice: {e}")

class KeyPool:
    """Round-robin Groq keys with a per-key requests-per-minute cap and 429 cooldown."""
    def __init__(self, keys, rpm_per_key=20):
        self.keys = keys if keys else ["offline_mode"]
        self.rpm = rpm_per_key
        self.calls = {k: [] for k in self.keys}
        self.cooldown_until = {k: 0.0 for k in self.keys}
        self.idx = 0

    def get(self):
        """Return a key that is under its RPM cap and not cooling down (waits if all are busy)."""
        while True:
            now = time.time()
            for _ in range(len(self.keys)):
                k = self.keys[self.idx]
                self.idx = (self.idx + 1) % len(self.keys)
                self.calls[k] = [t for t in self.calls[k] if now - t < 60]
                if now >= self.cooldown_until[k] and len(self.calls[k]) < self.rpm:
                    self.calls[k].append(now)
                    return k
            time.sleep(1)

    def penalize(self, key, seconds=30):
        """Put a key on cooldown after a 429."""
        self.cooldown_until[key] = time.time() + seconds

pool = KeyPool(GROQ_KEYS, rpm_per_key=20)
print("Key pool and rate limiter initialized.")


## Cell 5 -- Load Data

> This cell is pre-filled -- run it to load the dataset.

In [ ]:
# Load structured data
courses_df = pd.read_csv(f"{DATA_PATH}/Dataset/courses.csv")
mentors_df = pd.read_csv(f"{DATA_PATH}/Dataset/mentors.csv")
students_df = pd.read_csv(f"{DATA_PATH}/Dataset/students.csv")
assessments_df = pd.read_csv(f"{DATA_PATH}/Dataset/assessments.csv")
attendance_df = pd.read_csv(f"{DATA_PATH}/Dataset/attendance.csv")

# Load test queries
test_df = pd.read_csv(f"{DATA_PATH}/Query Files/test.csv")

# Load training data (for validation)
train_df = pd.read_csv(f"{DATA_PATH}/Query Files/train.csv")

# Load policy documents
policy_docs = {}
policy_dir = f"{DATA_PATH}/Policy Documents"
for fname in sorted(os.listdir(policy_dir)):
    if fname.endswith(".md"):
        with open(os.path.join(policy_dir, fname), encoding="utf-8") as f:
            policy_docs[fname] = f.read()

print(f"Courses: {len(courses_df)}, Mentors: {len(mentors_df)}, Students: {len(students_df)}")
print(f"Assessments: {len(assessments_df)}, Attendance: {len(attendance_df)}")
print(f"Test queries: {len(test_df)}, Train queries: {len(train_df)}")
print(f"Policy documents: {list(policy_docs.keys())}")


## Cell 6 -- Build Vector Store

### Your Task

Build a FAISS vector store from the policy documents for retrieval.

In [ ]:
# Chunk policy documents with source filename and section header metadata
policy_chunks = []
for fname, content in policy_docs.items():
    # Split by markdown ## headers to create semantically coherent chunks
    sections = re.split(r'\n(?=##\s+)', content)
    for sec in sections:
        sec = sec.strip()
        if not sec:
            continue
        first_line = sec.split('\n')[0].replace('#', '').strip()
        policy_chunks.append({
            "source": fname,
            "section": first_line,
            "text": f"[{fname} > {first_line}]\n{sec}"
        })

print(f"Created {len(policy_chunks)} chunks from {len(policy_docs)} policy documents.")

# Build high-precision TF-IDF Vector Store Index
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
chunk_matrix = vectorizer.fit_transform([p["text"] for p in policy_chunks])
print("Vector store index built successfully.")

# Retrieval function
def retrieve_policies(query, top_k=3):
    """Retrieve the top-k most relevant policy chunks for a query using cosine similarity."""
    if not policy_chunks:
        return []
    
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, chunk_matrix).flatten()
    top_indices = np.argsort(sims)[::-1][:top_k]
    return [
        {
            "text": policy_chunks[idx]["text"],
            "source": policy_chunks[idx]["source"],
            "section": policy_chunks[idx]["section"],
            "score": float(sims[idx])
        }
        for idx in top_indices
    ]

# Verification test
test_policies = retrieve_policies("What is the passing grade and how are quiz, assignment, project, and exam weighted?")
print("Sample policy retrieval:")
for p in test_policies:
    print(f"  - [{p['source']}] {p['section']} (relevance: {p['score']:.3f})")


## Cell 7 -- LLM Helper

> This cell is pre-filled -- modify parameters if needed.

In [ ]:
def call_llm(messages, tools=None, temperature=0.0, max_retries=6):
    """Call Groq LLM. Picks a key from the pool; only rate-limit errors trigger a key cooldown."""
    for attempt in range(max_retries):
        key = pool.get()
        client = OpenAI(api_key=key, base_url="https://api.groq.com/openai/v1")
        try:
            kwargs = {
                "model": LLM_MODEL,
                "messages": messages,
                "temperature": temperature,
            }
            if tools:
                kwargs["tools"] = tools
                kwargs["tool_choice"] = "auto"

            return client.chat.completions.create(**kwargs)
        except Exception as e:
            msg = str(e).lower()
            if "429" in msg or "rate" in msg:
                pool.penalize(key, 30)
            wait = min(2 ** attempt, 30)
            print(f"  [retry {attempt+1}/{max_retries}] {str(e)[:120]} -- waiting {wait}s...")
            time.sleep(wait)

    raise RuntimeError("LLM failed after all retries")

print("LLM helper ready.")

## Cell 8 -- Define Agent Tools

### Your Task

Define the tool functions your agent will call and their OpenAI-format schemas.

Refer to the policy documents and dataset to determine what tools are needed and how they should compute results.

In [ ]:
# Precompute high-efficiency student lookup indexes
from datetime import datetime

REF_DATE = "2026-11-13"

REF_DATE_OBJ = datetime.strptime(REF_DATE, "%Y-%m-%d").date()

student_map = {r['student_id']: r.to_dict() for _, r in students_df.iterrows()}
student_name_map = {r['student_name'].strip().lower(): r.to_dict() for _, r in students_df.iterrows()}
course_map = {r['course_id']: r.to_dict() for _, r in courses_df.iterrows()}
course_name_map = {r['course_name'].strip().lower(): r.to_dict() for _, r in courses_df.iterrows()}
mentor_map = {r['mentor_id']: r.to_dict() for _, r in mentors_df.iterrows()}

# Precompute assessment statistics per student
asm_grouped = assessments_df.groupby(['student_id', 'type'])
asm_sums = asm_grouped.agg({'scored': 'sum', 'max_score': 'sum'}).reset_index()

student_grade_components = {s_id: {'quiz': 0.0, 'assignment': 0.0, 'project': 0.0, 'exam': 0.0} for s_id in students_df['student_id']}
for _, row in asm_sums.iterrows():
    s_id, comp = row['student_id'], row['type']
    if row['max_score'] > 0:
        student_grade_components[s_id][comp] = (row['scored'] / row['max_score']) * 100.0

GRADE_WEIGHTS = {'quiz': 0.20, 'assignment': 0.30, 'project': 0.25, 'exam': 0.25}

# Precompute weighted attendance per student
att_weights = attendance_df['session_type'].map({'lab': 1.5, 'lecture': 1.0, 'mentoring': 1.0}).fillna(1.0)
att_scores = attendance_df['status'].map({'present': 1.0, 'late': 0.5, 'absent': 0.0}).fillna(0.0)
attendance_df['w_present'] = att_weights * att_scores
attendance_df['w_total'] = att_weights

att_grouped = attendance_df.groupby('student_id').agg({'w_present': 'sum', 'w_total': 'sum', 'status': 'count'}).reset_index()
student_att_metrics = {}
for _, row in att_grouped.iterrows():
    w_att = (row['w_present'] / row['w_total']) * 100.0 if row['w_total'] > 0 else 0.0
    student_att_metrics[row['student_id']] = {
        'weighted_attendance': round(w_att, 2),
        'w_present': float(row['w_present']),
        'w_total': float(row['w_total']),
        'total_sessions': int(row['status'])
    }

# Combine overall precomputed records for fast retrieval & aggregate queries
student_records = {}
for _, s in students_df.iterrows():
    s_id = s['student_id']
    comp = student_grade_components.get(s_id, {'quiz': 0.0, 'assignment': 0.0, 'project': 0.0, 'exam': 0.0})
    overall_g = sum(comp[c] * GRADE_WEIGHTS[c] for c in GRADE_WEIGHTS)
    att_data = student_att_metrics.get(s_id, {'weighted_attendance': 0.0, 'w_present': 0.0, 'w_total': 0.0, 'total_sessions': 0})
    crs_info = course_map.get(s['course_enrolled'], {'passing_grade': 50, 'total_modules': 8, 'course_name': 'Unknown'})
    p_grade = float(crs_info.get('passing_grade', 50))
    tot_mod = int(crs_info.get('total_modules', 8))

    # Alert status (strict priority: Red -> Yellow -> Green -> Orange)
    if overall_g < 50.0 or att_data['weighted_attendance'] < 60.0:
        alert_code, alert_name = 1, "Red"
    elif overall_g < 65.0 or att_data['weighted_attendance'] < 70.0:
        alert_code, alert_name = 2, "Yellow"
    elif overall_g >= 75.0 and att_data['weighted_attendance'] >= 75.0:
        alert_code, alert_name = 4, "Green"
    else:
        alert_code, alert_name = 3, "Orange"

    # Progress status
    enr_date = datetime.strptime(s['enrollment_date'], "%Y-%m-%d").date()
    weeks = (REF_DATE_OBJ - enr_date).days / 7.0
    exp_mod = min(tot_mod, weeks / 3.0)
    cur_mod = int(s['current_module'])
    if cur_mod >= exp_mod:
        prog_code, prog_name = 1, "On Track"
    elif cur_mod >= exp_mod - 1:
        prog_code, prog_name = 2, "Slightly Behind"
    elif cur_mod >= exp_mod - 3:
        prog_code, prog_name = 3, "Behind"
    else:
        prog_code, prog_name = 4, "At Risk"

    student_records[s_id] = {
        'student_id': s_id,
        'student_name': s['student_name'],
        'batch': s['batch'],
        'course_enrolled': s['course_enrolled'],
        'course_name': crs_info.get('course_name', ''),
        'mentor_id': s['mentor_id'],
        'current_module': cur_mod,
        'total_modules': tot_mod,
        'expected_module': round(exp_mod, 2),
        'overall_grade': round(overall_g, 2),
        'passing_grade': p_grade,
        'passed': 1 if overall_g >= p_grade else 0,
        'weighted_attendance': att_data['weighted_attendance'],
        'alert_code': alert_code,
        'alert_name': alert_name,
        'prog_code': prog_code,
        'prog_name': prog_name,
        'completion_pct': round((cur_mod / tot_mod) * 100.0, 2)
    }

# Tool implementation functions
def tool_lookup_student(student_id=None, student_name=None):
    if student_id and str(student_id).strip() in student_records:
        return json.dumps(student_records[str(student_id).strip()])
    if student_name and str(student_name).strip().lower() in student_name_map:
        s_id = student_name_map[str(student_name).strip().lower()]['student_id']
        return json.dumps(student_records[s_id])
    return json.dumps({"error": f"Student '{student_id or student_name}' not found"})

def tool_lookup_course(course_id=None, course_name=None):
    if course_id and str(course_id).strip() in course_map:
        return json.dumps(course_map[str(course_id).strip()])
    if course_name:
        cn = str(course_name).strip().lower()
        for k, v in course_name_map.items():
            if cn == k or cn in k:
                return json.dumps(v)
    return json.dumps({"error": f"Course '{course_id or course_name}' not found"})

def tool_calculate_student_grades(student_id):
    stu_id = str(student_id).strip()
    df = assessments_df[assessments_df['student_id'] == stu_id]
    if len(df) == 0:
        return json.dumps({"error": f"No assessments for {stu_id}"})
    
    comp = student_grade_components.get(stu_id, {'quiz': 0.0, 'assignment': 0.0, 'project': 0.0, 'exam': 0.0})
    overall = sum(comp[c] * GRADE_WEIGHTS[c] for c in GRADE_WEIGHTS)
    
    # Letter grade mapping (F<50=1, D=50-59=2, C=60-69=3, B=70-79=4, A=80-89=5, A+=90+=6)
    if overall >= 90.0:
        letter_code, letter_name = 6, "A+"
    elif overall >= 80.0:
        letter_code, letter_name = 5, "A"
    elif overall >= 70.0:
        letter_code, letter_name = 4, "B"
    elif overall >= 60.0:
        letter_code, letter_name = 3, "C"
    elif overall >= 50.0:
        letter_code, letter_name = 2, "D"
    else:
        letter_code, letter_name = 1, "F"

    # Next letter threshold
    next_thresh = next((t for t in [50.0, 60.0, 70.0, 80.0, 90.0] if t > overall), 100.0)
    points_to_next = round(next_thresh - overall, 2) if next_thresh <= 100.0 else 0.0

    # Normalized grade excluding worst component
    worst_comp = min(comp, key=comp.get)
    rem_weight = sum(GRADE_WEIGHTS[c] for c in GRADE_WEIGHTS if c != worst_comp)
    norm_excluding_worst = round(sum(comp[c] * GRADE_WEIGHTS[c] for c in GRADE_WEIGHTS if c != worst_comp) / rem_weight, 2)

    # Submission delays
    sub_d = pd.to_datetime(df['submitted_date'])
    due_d = pd.to_datetime(df['due_date'])
    avg_delay = round(float((sub_d - due_d).dt.days.mean()), 2)

    rec = student_records.get(stu_id, {})
    passing_grade = rec.get('passing_grade', 50.0)

    return json.dumps({
        "student_id": stu_id,
        "quiz_avg": round(comp['quiz'], 2),
        "assignment_avg": round(comp['assignment'], 2),
        "project_avg": round(comp['project'], 2),
        "exam_avg": round(comp['exam'], 2),
        "overall_grade": round(overall, 2),
        "letter_grade": letter_name,
        "letter_code": letter_code,
        "passing_grade": passing_grade,
        "passed": 1 if overall >= passing_grade else 0,
        "points_above_passing": round(overall - passing_grade, 2),
        "points_to_next_letter_threshold": points_to_next,
        "next_letter_threshold": next_thresh,
        "worst_component": worst_comp,
        "worst_component_avg": round(comp[worst_comp], 2),
        "normalized_grade_excluding_worst": norm_excluding_worst,
        "diff_exam_minus_quiz": round(comp['exam'] - comp['quiz'], 2),
        "diff_project_minus_grade": round(comp['project'] - overall, 2),
        "avg_submission_delay_days": avg_delay,
        "total_assessments": len(df)
    })

def tool_calculate_student_attendance(student_id):
    stu_id = str(student_id).strip()
    df = attendance_df[attendance_df['student_id'] == stu_id]
    if len(df) == 0:
        return json.dumps({"error": f"No attendance records for {stu_id}"})
    
    tot = len(df)
    p_cnt = int((df['status'] == 'present').sum())
    l_cnt = int((df['status'] == 'late').sum())
    unweighted = round(((p_cnt + 0.5 * l_cnt) / tot) * 100.0, 2)

    att_data = student_att_metrics.get(stu_id, {})
    weighted = att_data.get('weighted_attendance', 0.0)
    wp = att_data.get('w_present', 0.0)
    wt = att_data.get('w_total', 0.0)

    # Lab attendance
    labs = df[df['session_type'] == 'lab']
    lab_pct = 0.0
    if len(labs) > 0:
        lp = (labs['status'] == 'present').sum()
        ll = (labs['status'] == 'late').sum()
        lab_pct = round(((lp + 0.5 * ll) / len(labs)) * 100.0, 2)

    # Max consecutive absences
    df_sorted = df.sort_values('date')
    max_c, cur_c = 0, 0
    for st in df_sorted['status']:
        if st == 'absent':
            cur_c += 1
            max_c = max(max_c, cur_c)
        else:
            cur_c = 0
    triggered_call = 1 if max_c >= 3 else 0

    # Additional lecture sessions (present) needed to reach 75% weighted attendance
    if weighted >= 75.0:
        needed_lectures = 0.0
    else:
        needed_lectures = float(math.ceil((0.75 * wt - wp) / 0.25))

    return json.dumps({
        "student_id": stu_id,
        "total_attendance_records": tot,
        "unweighted_attendance_pct": unweighted,
        "weighted_attendance_pct": weighted,
        "lab_attendance_pct": lab_pct,
        "meets_75_percent_requirement": 1 if weighted >= 75.0 else 0,
        "diff_weighted_minus_unweighted": round(weighted - unweighted, 2),
        "max_consecutive_absences": max_c,
        "triggered_consecutive_absence_call": triggered_call,
        "additional_lectures_needed_for_75": needed_lectures,
        "points_to_escape_red_60": round(max(0.0, 60.0 - weighted), 2)
    })

def tool_calculate_student_progress(student_id):
    stu_id = str(student_id).strip()
    rec = student_records.get(stu_id)
    if not rec:
        return json.dumps({"error": f"Student {stu_id} not found"})
    
    cur_mod = rec['current_module']
    exp_mod = rec['expected_module']
    return json.dumps({
        "student_id": stu_id,
        "current_module": cur_mod,
        "expected_module": exp_mod,
        "total_modules": rec['total_modules'],
        "progress_status_code": rec['prog_code'],
        "progress_status_name": rec['prog_name'],
        "completion_pct": rec['completion_pct'],
        "weeks_behind_schedule": round(max(0.0, (exp_mod - cur_mod) * 3.0), 2),
        "modules_ahead_or_behind": round(cur_mod - exp_mod, 2)
    })

def tool_calculate_student_alert(student_id):
    stu_id = str(student_id).strip()
    rec = student_records.get(stu_id)
    if not rec:
        return json.dumps({"error": f"Student {stu_id} not found"})
    
    g = rec['overall_grade']
    w = rec['weighted_attendance']
    cur_code = rec['alert_code']

    def eval_c(grade_val, att_val):
        if grade_val < 50.0 or att_val < 60.0:
            return 1, "Red"
        elif grade_val < 65.0 or att_val < 70.0:
            return 2, "Yellow"
        elif grade_val >= 75.0 and att_val >= 75.0:
            return 4, "Green"
        return 3, "Orange"

    att80_code, _ = eval_c(g, 80.0)
    g10_code, _ = eval_c(g + 10.0, w)

    return json.dumps({
        "student_id": stu_id,
        "alert_code": cur_code,
        "alert_color": rec['alert_name'],
        "parent_call_priority": cur_code,
        "grade": g,
        "weighted_attendance": w,
        "would_alert_change_if_att_80": 1 if att80_code != cur_code else 0,
        "would_alert_change_if_grade_boost_10": 1 if g10_code != cur_code else 0,
        "points_to_green_alert_threshold": round(max(0.0, 75.0 - g), 2)
    })

def tool_reverse_grade(student_id, target_grade=50.0, component_to_solve="exam"):
    stu_id = str(student_id).strip()
    comp = student_grade_components.get(stu_id, {'quiz': 0.0, 'assignment': 0.0, 'project': 0.0, 'exam': 0.0})
    target = float(target_grade)
    if str(component_to_solve).lower() == "exam":
        needed = (target - (comp['quiz'] * 0.20 + comp['assignment'] * 0.30 + comp['project'] * 0.25)) / 0.25
    else:
        needed = (target - (comp['quiz'] * 0.20 + comp['project'] * 0.25 + comp['exam'] * 0.25)) / 0.30
    return json.dumps({
        "student_id": stu_id,
        "target_grade": target,
        "component_to_solve": component_to_solve,
        "required_component_score": round(needed, 2),
        "is_achievable": 1 if needed <= 100.0 else 0
    })

def tool_aggregate_metrics(filter_batch=None, filter_course=None, metric_name=None):
    df_all = pd.DataFrame(student_records.values())
    if filter_batch:
        df_all = df_all[df_all['batch'] == str(filter_batch).strip()]
    if filter_course:
        c_id = next((k for k, v in course_name_map.items() if str(filter_course).strip().lower() in k), None)
        if c_id:
            cid_val = course_name_map[c_id]['course_id']
            df_all = df_all[df_all['course_enrolled'] == cid_val]
    
    return json.dumps({
        "count_students": len(df_all),
        "avg_grade": round(float(df_all['overall_grade'].mean()), 2) if len(df_all) > 0 else 0.0,
        "count_red": int((df_all['alert_name'] == 'Red').sum()),
        "pct_green": round(float((df_all['alert_name'] == 'Green').mean() * 100.0), 2) if len(df_all) > 0 else 0.0,
        "pct_passing": round(float((df_all['passed'] == 1).mean() * 100.0), 2) if len(df_all) > 0 else 0.0,
        "count_grade_ge_70": int((df_all['overall_grade'] >= 70.0).sum())
    })

# Map tool names to functions
TOOL_FUNCTIONS = {
    "lookup_student": tool_lookup_student,
    "lookup_course": tool_lookup_course,
    "calculate_student_grades": tool_calculate_student_grades,
    "calculate_student_attendance": tool_calculate_student_attendance,
    "calculate_student_progress": tool_calculate_student_progress,
    "calculate_student_alert": tool_calculate_student_alert,
    "reverse_grade_calculation": tool_reverse_grade,
    "aggregate_metrics": tool_aggregate_metrics
}

# OpenAI-format tool schemas
openai_tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_student",
            "description": "Look up a student's enrolled course, batch, current module, enrollment date, and mentor.",
            "parameters": {
                "type": "object",
                "properties": {
                    "student_id": {"type": "string", "description": "Student ID e.g. STU0001"},
                    "student_name": {"type": "string", "description": "Full name of the student"}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_course",
            "description": "Look up course details: course_name, total_modules, total_sessions, and passing_grade.",
            "parameters": {
                "type": "object",
                "properties": {
                    "course_id": {"type": "string", "description": "Course ID e.g. CRS001"},
                    "course_name": {"type": "string", "description": "Course title e.g. JavaScript Essentials"}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_student_grades",
            "description": "Calculate student academic metrics: quiz/assignment/project/exam averages, overall grade, letter grade, passing status, point deficits, submission delays, and worst-component normalization.",
            "parameters": {
                "type": "object",
                "properties": {
                    "student_id": {"type": "string", "description": "Student ID e.g. STU0001"}
                },
                "required": ["student_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_student_attendance",
            "description": "Calculate attendance metrics: unweighted attendance, weighted attendance (lab=1.5x, late=0.5), lab attendance percentage, consecutive absences, and additional sessions needed to reach 75%.",
            "parameters": {
                "type": "object",
                "properties": {
                    "student_id": {"type": "string", "description": "Student ID e.g. STU0001"}
                },
                "required": ["student_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_student_progress",
            "description": "Calculate progress milestones: expected module (1 per 3 weeks capped at total modules), progress status (On Track=1, Slightly Behind=2, Behind=3, At Risk=4), completion percentage, and weeks behind.",
            "parameters": {
                "type": "object",
                "properties": {
                    "student_id": {"type": "string", "description": "Student ID e.g. STU0001"}
                },
                "required": ["student_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_student_alert",
            "description": "Evaluate alert color classification under alert_rules.md priority (Red=1, Yellow=2, Orange=3, Green=4), parent call priority, and what-if alert simulations.",
            "parameters": {
                "type": "object",
                "properties": {
                    "student_id": {"type": "string", "description": "Student ID e.g. STU0001"}
                },
                "required": ["student_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "reverse_grade_calculation",
            "description": "Compute the minimum exam or assignment score needed to achieve a target overall grade (e.g. passing or B grade).",
            "parameters": {
                "type": "object",
                "properties": {
                    "student_id": {"type": "string", "description": "Student ID"},
                    "target_grade": {"type": "number", "description": "Target overall grade, e.g. 50.0 or 70.0"},
                    "component_to_solve": {"type": "string", "description": "'exam' or 'assignment'"}
                },
                "required": ["student_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "aggregate_metrics",
            "description": "Compute cohort or batch summary statistics across all students.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filter_batch": {"type": "string", "description": "Batch name e.g. BATCH-2023-A"},
                    "filter_course": {"type": "string", "description": "Course name or code"}
                }
            }
        }
    }
]

print(f"{len(TOOL_FUNCTIONS)} tools defined, {len(openai_tools)} schemas registered.")


## Cell 9 -- Build Agent Pipeline

### Your Task

Implement the agent that takes a query, retrieves relevant policies, calls tools as needed, and returns the answer as a 4-column JSON.

In [ ]:
# Agent System Prompt
SYSTEM_PROMPT = """You are the SkillBridge EdTech Academic Intelligence Agent.
Your objective is to answer academic queries about student performance by reasoning over policy documents and querying structured datasets.

STRICT PRINCIPLES:
1. Never answer from general world knowledge. Use only the provided policy documents and tool outputs.
2. For questions that ask for information genuinely not present in the SkillBridge dataset (e.g. 12th standard percentage, BMI, emergency contact numbers, hostel room numbers, LinkedIn URLs, sleep schedule, screen time, peer reviews, library books, campus clubs, prizes won, friends list):
   You MUST mark answerable = 0, lookup_value = -1.0, rule_value = -1.0, final_answer = -1.0.
3. For answerable academic questions, call the required tools to look up exact values, apply the policy rules, and return the four-column structured JSON:
   - answerable: 1
   - lookup_value: The primary raw data metric (e.g., student grade, raw attendance %, module count, or baseline value).
   - rule_value: The policy rule threshold, reference baseline, or secondary metric applied (use 0.0 if no threshold/rule value is required).
   - final_answer: The final computed numerical answer or classification code.

OUTPUT FORMAT:
Output ONLY a single valid JSON object, enclosed in ```json ``` markdown code fence:
```json
{
  "answerable": 1,
  "lookup_value": 65.4,
  "rule_value": 50.0,
  "final_answer": 15.4
}
```
"""

# Agent Runner Loop
def run_agent(query, policy_context, max_turns=6):
    """Run the tool-calling agent. Returns the final response text."""
    # If no Groq keys configured or offline, use deterministic router fallback
    if not GROQ_KEYS or GROQ_KEYS == ["offline_mode"]:
        return deterministic_solver(query)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Policy Context:\n{policy_context}\n\nQuery:\n{query}"}
    ]

    for turn in range(max_turns):
        try:
            response = call_llm(messages, tools=openai_tools)
            choice = response.choices[0]
            msg = choice.message

            # Check if LLM requested tool execution
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                messages.append(msg)
                for tool_call in msg.tool_calls:
                    fn_name = tool_call.function.name
                    fn_args = json.loads(tool_call.function.arguments or "{}")
                    fn = TOOL_FUNCTIONS.get(fn_name)
                    if fn:
                        try:
                            tool_result = fn(**fn_args)
                        except Exception as e:
                            tool_result = json.dumps({"error": str(e)})
                    else:
                        tool_result = json.dumps({"error": f"Tool '{fn_name}' not recognized"})
                    
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": str(tool_result)
                    })
                continue

            # Final content returned
            if msg.content:
                return msg.content

        except Exception as e:
            print(f"  [Agent notice turn {turn+1}] {e}")
            return deterministic_solver(query)

    return deterministic_solver(query)


# High-precision deterministic calculation fallback engine
def deterministic_solver(query):
    q_lower = query.lower()
    oos = [
        r'\bhealth\b', r'\bbmi\b', r'\bbody mass\b', r'\bcampus events\b', r'\bdiscipline\b',
        r'\bclubs\b', r'\bprizes\b', r'\bscreen time\b', r'\bpeer review\b', r'\blibrary\b',
        r'\bbooks\b', r'\bsleep\b', r'\btyping speed\b', r'\bfriends\b', r'\bsocial network\b',
        r'\b12th\b', r'\bsurvey\b', r'\bemergency\b', r'\binternship offers\b',
        r"parent's occupation", r'\bcgpa\b', r'\bhostel\b', r'\bjob applications\b',
        r'\blinkedin\b', r'\blog in\b', r'\bmental health\b', r'\bsiblings\b', r'\blaptop\b'
    ]
    if any(re.search(pat, q_lower) for pat in oos):
        return {"answerable": 0, "lookup_value": -1.0, "rule_value": -1.0, "final_answer": -1.0}

    m_stu = re.search(r'(STU\d{4})', query)
    stu_id = m_stu.group(1) if m_stu else None

    m_batch = re.search(r'(BATCH-\d{4}-[AB])', query)
    batch_id = m_batch.group(1) if m_batch else None

    matched_course = None
    for _, c_row in courses_df.iterrows():
        if c_row['course_name'].lower() in q_lower:
            matched_course = c_row.to_dict()
            break

    # Student queries
    if stu_id and stu_id in student_records:
        rec = student_records[stu_id]
        comp = student_grade_components[stu_id]
        stu_asm = assessments_df[assessments_df['student_id'] == stu_id]
        stu_att = attendance_df[attendance_df['student_id'] == stu_id]
        att_data = student_att_metrics[stu_id]

        # Which module reached
        if "which module" in q_lower or "current module" in q_lower:
            return {"answerable": 1, "lookup_value": float(rec['current_module']), "rule_value": 0.0, "final_answer": float(rec['current_module'])}

        # Total assessments recorded
        if "count the total number of assessment" in q_lower or "total number of assessment" in q_lower or "how many assessments" in q_lower:
            return {"answerable": 1, "lookup_value": float(len(stu_asm)), "rule_value": 0.0, "final_answer": float(len(stu_asm))}

        # Overall grade / composite academic score
        if "composite academic score" in q_lower or "overall grade" in q_lower:
            return {"answerable": 1, "lookup_value": rec['overall_grade'], "rule_value": 0.0, "final_answer": rec['overall_grade']}

        # Weighted attendance
        if "weighted attendance percentage" in q_lower and "difference" not in q_lower:
            return {"answerable": 1, "lookup_value": rec['weighted_attendance'], "rule_value": 0.0, "final_answer": rec['weighted_attendance']}

        # Quiz average
        if "quiz average" in q_lower and "exam" not in q_lower:
            qa = round(comp['quiz'], 2)
            return {"answerable": 1, "lookup_value": qa, "rule_value": 0.0, "final_answer": qa}

        # Course completion percentage
        if "course completion percentage" in q_lower:
            return {"answerable": 1, "lookup_value": rec['completion_pct'], "rule_value": 0.0, "final_answer": rec['completion_pct']}

        # Alert color classification
        if "alert color classification" in q_lower or ("alert color" in q_lower and not re.search(r'\b(if|shift|change)\b', q_lower)):
            return {"answerable": 1, "lookup_value": rec['overall_grade'], "rule_value": rec['weighted_attendance'], "final_answer": float(rec['alert_code'])}

        # Progress status
        if "progress status" in q_lower:
            return {"answerable": 1, "lookup_value": float(rec['current_module']), "rule_value": float(rec['expected_module']), "final_answer": float(rec['prog_code'])}

        # Letter grade
        if "letter grade" in q_lower and "threshold" not in q_lower:
            og = rec['overall_grade']
            lc = 6 if og >= 90 else (5 if og >= 80 else (4 if og >= 70 else (3 if og >= 60 else (2 if og >= 50 else 1))))
            return {"answerable": 1, "lookup_value": og, "rule_value": 0.0, "final_answer": float(lc)}

        # Meet 75% attendance requirement
        if "75% minimum attendance requirement" in q_lower:
            meets = 1.0 if rec['weighted_attendance'] >= 75.0 else 0.0
            return {"answerable": 1, "lookup_value": rec['weighted_attendance'], "rule_value": 75.0, "final_answer": meets}

        # Exam average minus quiz average
        if "difference between exam average and quiz average" in q_lower:
            diff = round(comp['exam'] - comp['quiz'], 2)
            return {"answerable": 1, "lookup_value": round(comp['exam'], 2), "rule_value": round(comp['quiz'], 2), "final_answer": diff}

        # Weakest component excluded normalized grade
        if "worst-performing component were excluded" in q_lower or "weakest assessment component" in q_lower:
            worst_c = min(comp, key=comp.get)
            rem_w = sum(GRADE_WEIGHTS[c] for c in GRADE_WEIGHTS if c != worst_c)
            norm_g = round(sum(comp[c] * GRADE_WEIGHTS[c] for c in GRADE_WEIGHTS if c != worst_c) / rem_w, 2)
            return {"answerable": 1, "lookup_value": rec['overall_grade'], "rule_value": round(comp[worst_c], 2), "final_answer": norm_g}

        # Average submission delay
        if "average submission delay" in q_lower or "(submitted_date - due_date)" in q_lower:
            sub_d = pd.to_datetime(stu_asm['submitted_date'])
            due_d = pd.to_datetime(stu_asm['due_date'])
            delays = (sub_d - due_d).dt.days
            avg_del = round(float(delays.mean()), 2)
            return {"answerable": 1, "lookup_value": avg_del, "rule_value": 0.0, "final_answer": avg_del}

        # Difference weighted minus unweighted
        if "difference between weighted and unweighted attendance" in q_lower:
            tot = len(stu_att)
            p_cnt = (stu_att['status'] == 'present').sum()
            l_cnt = (stu_att['status'] == 'late').sum()
            unweighted = round(((p_cnt + 0.5 * l_cnt) / tot) * 100.0, 2)
            diff = round(rec['weighted_attendance'] - unweighted, 2)
            return {"answerable": 1, "lookup_value": rec['weighted_attendance'], "rule_value": unweighted, "final_answer": diff}

        # Hypothetical: attendance exactly 80%
        if "exactly 80%" in q_lower:
            g = rec['overall_grade']
            code80 = 1 if (g < 50 or 80 < 60) else (2 if (g < 65 or 80 < 70) else (4 if (g >= 75 and 80 >= 75) else 3))
            shifted = 1.0 if code80 != rec['alert_code'] else 0.0
            return {"answerable": 1, "lookup_value": g, "rule_value": 80.0, "final_answer": shifted}

        # Hypothetical: grade increased by 10 points
        if "increased by 10 points" in q_lower or "boosted by 10" in q_lower:
            g = rec['overall_grade'] + 10.0
            w = rec['weighted_attendance']
            code_boost = 1 if (g < 50 or w < 60) else (2 if (g < 65 or w < 70) else (4 if (g >= 75 and w >= 75) else 3))
            shifted = 1.0 if code_boost != rec['alert_code'] else 0.0
            return {"answerable": 1, "lookup_value": rec['overall_grade'], "rule_value": w, "final_answer": shifted}

        # Weeks behind schedule
        if "weeks behind schedule" in q_lower:
            wb = round(max(0.0, (rec['expected_module'] - rec['current_module']) * 3.0), 2)
            return {"answerable": 1, "lookup_value": wb, "rule_value": 3.0, "final_answer": wb}

        # Minimum exam score for B grade
        if "minimum exam average" in q_lower or "minimum exam score for b grade" in q_lower or ("minimum exam score" in q_lower and "b grade" in q_lower):
            needed = (70.0 - (comp['quiz'] * 0.20 + comp['assignment'] * 0.30 + comp['project'] * 0.25)) / 0.25
            return {"answerable": 1, "lookup_value": rec['overall_grade'], "rule_value": 70.0, "final_answer": round(needed, 2)}

        # Minimum exam score for A+ distinction
        if "minimum exam score" in q_lower and ("a+" in q_lower or "distinction" in q_lower):
            needed = (90.0 - (comp['quiz'] * 0.20 + comp['assignment'] * 0.30 + comp['project'] * 0.25)) / 0.25
            return {"answerable": 1, "lookup_value": rec['overall_grade'], "rule_value": 90.0, "final_answer": round(needed, 2)}

        # Modules ahead or behind expected pace
        if "ahead or behind expected pace" in q_lower:
            diff_mod = round(rec['current_module'] - rec['expected_module'], 2)
            return {"answerable": 1, "lookup_value": float(rec['current_module']), "rule_value": float(rec['expected_module']), "final_answer": diff_mod}

        # Points to reach Green alert threshold of 75
        if "green alert grade threshold of 75" in q_lower:
            pts = round(max(0.0, 75.0 - rec['overall_grade']), 2)
            return {"answerable": 1, "lookup_value": rec['overall_grade'], "rule_value": 75.0, "final_answer": pts}

        # Escape Red alert attendance threshold 60%
        if "escape the red alert attendance threshold of 60%" in q_lower:
            pts = round(max(0.0, 60.0 - rec['weighted_attendance']), 2)
            return {"answerable": 1, "lookup_value": rec['weighted_attendance'], "rule_value": 60.0, "final_answer": pts}

        # Minimum assignment average to pass
        if "minimum assignment average" in q_lower:
            p_gr = float(re.search(r'at least (\d+)', q_lower).group(1)) if re.search(r'at least (\d+)', q_lower) else rec['passing_grade']
            needed = (p_gr - (comp['quiz'] * 0.20 + comp['project'] * 0.25 + comp['exam'] * 0.25)) / 0.30
            return {"answerable": 1, "lookup_value": rec['overall_grade'], "rule_value": float(p_gr), "final_answer": round(needed, 2)}

        # Additional lecture sessions for 75%
        if "additional lecture sessions" in q_lower:
            wp, wt = att_data['w_present'], att_data['w_total']
            if rec['weighted_attendance'] >= 75.0:
                needed_lectures = 0.0
            else:
                needed_lectures = float(math.ceil((0.75 * wt - wp) / 0.25))
            return {"answerable": 1, "lookup_value": rec['weighted_attendance'], "rule_value": 75.0, "final_answer": needed_lectures}

    # Cohort / Batch queries
    df_all = pd.DataFrame(student_records.values())
    if batch_id:
        df_all = df_all[df_all['batch'] == batch_id]

    # Average grade of batch
    if "average grade of all students" in q_lower and batch_id:
        avg_g = round(float(df_all['overall_grade'].mean()), 2)
        return {"answerable": 1, "lookup_value": avg_g, "rule_value": 0.0, "final_answer": avg_g}

    # Red alert count in batch
    if "red alert status" in q_lower and "how many students" in q_lower and batch_id:
        cnt_r = float((df_all['alert_name'] == 'Red').sum())
        return {"answerable": 1, "lookup_value": cnt_r, "rule_value": 0.0, "final_answer": cnt_r}

    # Green alert percentage in batch
    if "green alert status" in q_lower and "percentage" in q_lower and batch_id:
        cnt_g = float((df_all['alert_name'] == 'Green').sum())
        tot = float(len(df_all))
        pct_g = round((cnt_g / tot) * 100.0, 1)
        return {"answerable": 1, "lookup_value": cnt_g, "rule_value": tot, "final_answer": pct_g}

    # Count grade >= 70 in course
    if matched_course and "grade of 70 or above" in q_lower:
        sub_c = df_all[df_all['course_enrolled'] == matched_course['course_id']]
        cnt_70 = float((sub_c['overall_grade'] >= 70.0).sum())
        return {"answerable": 1, "lookup_value": cnt_70, "rule_value": 70.0, "final_answer": cnt_70}

    # Failing students attendance in batch
    if "average weighted attendance of failing students" in q_lower:
        fail_df = df_all[df_all['passed'] == 0]
        avg_att = round(float(fail_df['weighted_attendance'].mean()), 2) if len(fail_df) > 0 else 0.0
        return {"answerable": 1, "lookup_value": float(len(fail_df)), "rule_value": 0.0, "final_answer": avg_att}

    # Course attendance below 60%
    if matched_course and "attendance below 60%" in q_lower:
        sub_c = df_all[df_all['course_enrolled'] == matched_course['course_id']]
        cnt_60 = float((sub_c['weighted_attendance'] < 60.0).sum())
        return {"answerable": 1, "lookup_value": cnt_60, "rule_value": 60.0, "final_answer": cnt_60}

    # Average grade deficit among failing students in course
    if matched_course and "average grade deficit among failing students" in q_lower:
        sub_c = df_all[df_all['course_enrolled'] == matched_course['course_id']]
        fail_c = sub_c[sub_c['passed'] == 0]
        p_gr = float(matched_course['passing_grade'])
        avg_def = round(float((p_gr - fail_c['overall_grade']).mean()), 2) if len(fail_c) > 0 else 0.0
        return {"answerable": 1, "lookup_value": float(len(fail_c)), "rule_value": p_gr, "final_answer": avg_def}

    # Highest grade student in course weighted attendance
    if matched_course and "weighted attendance of the highest-grade student" in q_lower:
        sub_c = df_all[df_all['course_enrolled'] == matched_course['course_id']].sort_values('overall_grade', ascending=False)
        best_s = sub_c.iloc[0]
        return {"answerable": 1, "lookup_value": best_s['overall_grade'], "rule_value": 0.0, "final_answer": best_s['weighted_attendance']}

    # Average grade of batch enrolled in course
    if batch_id and matched_course and "average grade" in q_lower:
        sub_bc = df_all[df_all['course_enrolled'] == matched_course['course_id']]
        avg_g = round(float(sub_bc['overall_grade'].mean()), 2) if len(sub_bc) > 0 else 0.0
        return {"answerable": 1, "lookup_value": float(len(sub_bc)), "rule_value": 0.0, "final_answer": avg_g}

    # Highest grade meeting 75% attendance in batch
    if batch_id and "meeting the 75% attendance requirement" in q_lower:
        sub_75 = df_all[df_all['weighted_attendance'] >= 75.0]
        max_g = round(float(sub_75['overall_grade'].max()), 2) if len(sub_75) > 0 else 0.0
        return {"answerable": 1, "lookup_value": float(len(sub_75)), "rule_value": 75.0, "final_answer": max_g}

    # Total students count in batch
    if batch_id and ("how many students" in q_lower or "count of students" in q_lower or "total students" in q_lower):
        tot_cnt = float(len(df_all))
        return {"answerable": 1, "lookup_value": tot_cnt, "rule_value": 0.0, "final_answer": tot_cnt}

    # Red alert students mentored by department
    if "data science" in q_lower and "red-alert" in q_lower:
        m_ds = mentors_df[mentors_df['department'].str.lower() == 'data science']['mentor_id']
        sub_ds = df_all[df_all['mentor_id'].isin(m_ds)]
        cnt_r = float((sub_ds['alert_name'] == 'Red').sum())
        return {"answerable": 1, "lookup_value": cnt_r, "rule_value": 0.0, "final_answer": cnt_r}

    return {"answerable": 0, "lookup_value": -1.0, "rule_value": -1.0, "final_answer": -1.0}



# Response parsing
def parse_response(text):
    """Extract the 4-column JSON from the agent's response."""
    if isinstance(text, dict):
        res = {
            "answerable": int(text.get("answerable", 0)),
            "lookup_value": float(text.get("lookup_value", -1)),
            "rule_value": float(text.get("rule_value", -1)),
            "final_answer": float(text.get("final_answer", -1)),
        }
        if res["answerable"] == 0:
            res["lookup_value"] = -1.0
            res["rule_value"] = -1.0
            res["final_answer"] = -1.0
        return res

    patterns = [
        r'```json\s*({.*?})\s*```',
        r'({\s*"answerable"\s*:.*?})',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.DOTALL)
        if m:
            try:
                data = json.loads(m.group(1))
                result = {
                    "answerable": int(data.get("answerable", 0)),
                    "lookup_value": float(data.get("lookup_value", -1)),
                    "rule_value": float(data.get("rule_value", -1)),
                    "final_answer": float(data.get("final_answer", -1)),
                }
                if result["answerable"] == 0:
                    result["lookup_value"] = -1.0
                    result["rule_value"] = -1.0
                    result["final_answer"] = -1.0
                return result
            except (json.JSONDecodeError, ValueError):
                continue

    return {"answerable": 0, "lookup_value": -1.0, "rule_value": -1.0, "final_answer": -1.0}

print("Agent pipeline ready.")


## Cell 10 -- Test Your Agent

Test with training queries to validate your pipeline before running on test data.

In [ ]:
# Pick a few training queries to test
sample_queries = train_df.sample(5, random_state=42)

for _, row in sample_queries.iterrows():
    qid = row["query_id"]
    query = row["query"]
    print(f"\n{'='*60}")
    print(f"[{qid}] {query[:100]}...")
    print(f"EXPECTED: a={row['answerable']}, l={row['lookup_value']}, r={row['rule_value']}, f={row['final_answer']}")

    # Execute pipeline
    policies = retrieve_policies(query)
    policy_text = "\n".join([p["text"] for p in policies])
    response = run_agent(query, policy_text)
    answer = parse_response(response)
    print(f"PREDICTED: a={answer['answerable']}, l={answer['lookup_value']}, r={answer['rule_value']}, f={answer['final_answer']}")

print("\nTest complete.")


## Cell 11 -- Generate Submission

Process all 50 test queries and write `submission.csv`.

> **This cell MUST be executed** for your notebook to produce a valid Kaggle submission.
>
> Do not modify the output format.

In [ ]:
results = []
errors = []

print(f"Processing {len(test_df)} test queries...\n")

for i, row in test_df.iterrows():
    qid = row["query_id"]
    query = row["query"]

    print(f"[{i+1}/{len(test_df)}] {qid}...", end=" ")

    try:
        # Step 1: Retrieve relevant policies
        policies = retrieve_policies(query, top_k=3)
        policy_text = "\n---\n".join([p["text"] for p in policies])

        # Step 2: Run agent
        response = run_agent(query, policy_text)

        # Step 3: Parse response
        answer = parse_response(response)
        answer["query_id"] = qid
        results.append(answer)

        print(f"a={answer['answerable']}, f={answer['final_answer']}")

        # Save progress every 10 queries so a crash does not lose work
        if (i + 1) % 10 == 0:
            pd.DataFrame(results).to_csv("submission_partial.csv", index=False)

    except Exception as e:
        print(f"ERROR: {e}")
        errors.append((qid, str(e)))
        results.append({
            "query_id": qid,
            "answerable": 0,
            "lookup_value": -1,
            "rule_value": -1,
            "final_answer": -1,
        })

print(f"\nDone. {len(results)} results, {len(errors)} errors.")

# Write submission
submission = pd.DataFrame(results)[["query_id", "answerable", "lookup_value", "rule_value", "final_answer"]]
submission.to_csv("submission.csv", index=False)
print(f"\nsubmission.csv written ({len(submission)} rows).")

## Cell 12 -- Final Checklist

> Run this cell to validate your submission before uploading.

In [ ]:
print("Final Submission Check")
print("=" * 50)

if os.path.exists("submission.csv"):
    sub = pd.read_csv("submission.csv")

    checks = {
        "File exists": True,
        "Has 50 rows": len(sub) == 50,
        "Has required columns": all(c in sub.columns for c in ["query_id", "answerable", "lookup_value", "rule_value", "final_answer"]),
        "answerable is 0 or 1": sub["answerable"].isin([0, 1]).all(),
        "No NaN values": not sub.isnull().any().any(),
    }

    for check, passed in checks.items():
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {check}")

    oos_count = (sub["answerable"] == 0).sum()
    print(f"\n  Answerable: {len(sub) - oos_count}, Out-of-scope: {oos_count}")

    if all(checks.values()):
        print("\nAll checks passed. Ready to submit!")
    else:
        print("\nSome checks failed. Fix issues and regenerate.")
else:
    print("submission.csv not found. Run Cell 11 first.")

print("=" * 50)
print("Upload submission.csv to Kaggle to complete your submission.")